# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imnxr/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
import os
import duckdb
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd().parents[1] / ".env"
load_dotenv(env_path)

hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        f"HF_TOKEN was not found. Expected .env at: {env_path}"
    )
con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
    """
)

REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/'
    'fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("DuckDB connection created.")
print("Using the March 2026 warehouse partition.")

DuckDB connection created.
Using the March 2026 warehouse partition.


In [2]:
window_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count
    FROM {REL}
""").df()

window_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,earliest_date,latest_date,client_count,content_count
0,9841378,2026-03-01,2026-03-31,55,331437


In [4]:
from pathlib import Path

repo_root = Path.cwd().parents[1]
temp_dir = repo_root / "work" / "outputs" / "duckdb_temp"
temp_dir.mkdir(parents=True, exist_ok=True)

con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 2")
con.execute(f"SET temp_directory = '{temp_dir.as_posix()}'")

print("DuckDB temporary storage configured:", temp_dir)

DuckDB temporary storage configured: d:\Internship\flyrank-ml-internship\work\outputs\duckdb_temp


In [4]:
import pandas as pd

duplicate_days = []

for day in pd.date_range("2026-03-01", "2026-03-31"):
    day_string = day.strftime("%Y-%m-%d")

    duplicates = con.sql(f"""
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS duplicate_count
        FROM {REL}
        WHERE report_date = DATE '{day_string}'
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
        HAVING COUNT(*) > 1
        LIMIT 5
    """).df()

    if not duplicates.empty:
        duplicate_days.append(duplicates)
        print(f"Duplicates found on {day_string}")
    else:
        print(f"{day_string}: no duplicates")

if duplicate_days:
    grain_check = pd.concat(duplicate_days, ignore_index=True)
else:
    grain_check = pd.DataFrame(
        columns=[
            "report_date",
            "client_hash_id",
            "content_hash_id",
            "duplicate_count",
        ]
    )

grain_check

2026-03-01: no duplicates
2026-03-02: no duplicates
2026-03-03: no duplicates
2026-03-04: no duplicates
2026-03-05: no duplicates
2026-03-06: no duplicates
2026-03-07: no duplicates
2026-03-08: no duplicates
2026-03-09: no duplicates
2026-03-10: no duplicates
2026-03-11: no duplicates
2026-03-12: no duplicates
2026-03-13: no duplicates
2026-03-14: no duplicates
2026-03-15: no duplicates
2026-03-16: no duplicates
2026-03-17: no duplicates
2026-03-18: no duplicates
2026-03-19: no duplicates
2026-03-20: no duplicates
2026-03-21: no duplicates
2026-03-22: no duplicates
2026-03-23: no duplicates
2026-03-24: no duplicates
2026-03-25: no duplicates
2026-03-26: no duplicates
2026-03-27: no duplicates
2026-03-28: no duplicates
2026-03-29: no duplicates
2026-03-30: no duplicates
2026-03-31: no duplicates


,report_date,client_hash_id,content_hash_id,duplicate_count


### Unit of analysis and time window

One row represents one content item's daily performance for one client on one report date.

The unit key is:

`report_date × client_hash_id × content_hash_id`

For this assignment, I use the March 2026 partition of `fact_content_daily_performance` as a mid-panel development window.

The March slice contains 9,841,378 rows, covers 2026-03-01 through 2026-03-31, includes 55 clients, and includes 331,437 content items.

The grain check found no duplicate unit keys on any day in March 2026, which supports the stated row definition.

The June 2026 sample is excluded from development because it is the final month and should remain a sealed future evaluation window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
schema_check = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {REL}
""").df()

schema_check

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


### Field classification

#### Features

These fields are observable before the decision moment and can be used to describe March 2026 performance:

- `gsc_impressions` — observed Google Search impressions.
- `gsc_clicks` — observed Google Search clicks.
- `gsc_avg_position` — observed average Google Search ranking position.
- `ga4_sessions` — observed website sessions, only where GA4 data is available.
- `ga4_engaged_sessions` — observed engaged sessions, only where GA4 data is available.

Only a maximum of five features will be used in the final feature frame.

#### Label / proxy

The label is `april_click_label`.

It equals 1 when a content item receives at least one Google Search click
during April 2026, and 0 when it receives no Google Search clicks during
April 2026.

`april_gsc_clicks` and `april_click_label` belong to the future outcome
window. They must not be included in the March feature set.

#### Context

These fields are used for identifying, grouping, filtering, joining, or interpreting rows, but not as model features:

- `report_date`
- `client_hash_id`
- `content_hash_id`
- `month`
- `client_has_gsc`
- `client_has_ga4`
- `gsc_data_available`
- `ga4_data_available`

The hash IDs are pseudonymous identifiers, not meaningful predictive measurements.

#### Excluded

- `gsc_sum_position` — excluded because it is an intermediate aggregate used to calculate average position and is not directly interpretable as an independent feature.
- `ga4_pageviews` — excluded to avoid adding a near-duplicate traffic-volume measure alongside sessions.
- `ga4_users` — excluded to keep the feature set small and avoid another strongly overlapping traffic-volume measure.
- `ga4_total_engagement_sec` — excluded because it is heavily influenced by traffic volume unless normalized.
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, and `sessions_paid` — excluded from this first contract to keep the model limited to five clear features.
- `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, and `ai_other` — excluded because these sparse source-specific counts are outside the current future-click classification question.
- `scroll_events` — excluded because it is a raw event count influenced by traffic volume and is not normalized.

In [6]:
feature_quality_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0.0 END)
            AS gsc_impressions_missing_rate,

        AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0.0 END)
            AS gsc_clicks_missing_rate,

        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0.0 END)
            AS gsc_avg_position_missing_rate,

        AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0.0 END)
            AS ga4_sessions_missing_rate,

        AVG(CASE WHEN ga4_engaged_sessions IS NULL THEN 1.0 ELSE 0.0 END)
            AS ga4_engaged_sessions_missing_rate,

        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
            AS rows_with_gsc_data,

        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
            AS rows_with_ga4_data

    FROM {REL}
""").df()

feature_quality_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_impressions_missing_rate,gsc_clicks_missing_rate,gsc_avg_position_missing_rate,ga4_sessions_missing_rate,ga4_engaged_sessions_missing_rate,rows_with_gsc_data,rows_with_ga4_data
0,9841378,0.0,0.0,0.633074,0.30674,0.30674,3611061,413966


### Feature availability result

The March 2026 slice contains 9,841,378 rows.

- `gsc_impressions` missing rate: 0.0%
- `gsc_clicks` missing rate: 0.0%
- `gsc_avg_position` missing rate: 63.31%
- `ga4_sessions` missing rate: 30.67%
- `ga4_engaged_sessions` missing rate: 30.67%
- Rows with `gsc_data_available IS TRUE`: 3,611,061
- Rows with `ga4_data_available IS TRUE`: 413,966

The missingness is not random. Search and analytics fields depend on whether GSC or GA4 data was available for that client and date. Therefore, availability flags must be checked before interpreting zero or missing values.

In [7]:
usable_rows_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM {REL}
""").df()

usable_rows_check

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


### Usable-row result

Out of 9,841,378 March rows:

- 3,611,061 have `gsc_data_available IS TRUE`
- 413,966 have `ga4_data_available IS TRUE`
- 364,347 have both GSC and GA4 available

For the combined five-feature frame, I will use only rows where both availability flags are true. This prevents unavailable GA4 or GSC history from being mistaken for real zero performance.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
final_slice_check = con.sql(f"""
    SELECT
        COUNT(*) AS usable_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count
    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""").df()

final_slice_check

,usable_rows,earliest_date,latest_date,client_count,content_count
0,364347,2026-03-01,2026-03-31,34,63856


### Final usable slice

After requiring both `gsc_data_available IS TRUE` and `ga4_data_available IS TRUE`, the usable March 2026 slice contains:

- 364,347 rows
- 34 clients
- 63,856 content items
- dates from 2026-03-01 through 2026-03-31

This is the slice used for the combined five-feature analysis.

In [9]:
feature_frame = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_engaged_sessions
    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,0,5.400000,1,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,0,5.666667,2,0
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,0,5.156425,2,0
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,0,7.694444,1,0
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,1,6.167885,1,0


### Five-feature frame

The model-facing feature set contains exactly five observed March 2026 measurements:

1. `gsc_impressions` — search visibility observed before the future outcome window.
2. `gsc_clicks` — search traffic observed before the future outcome window.
3. `gsc_avg_position` — average search ranking observed during March.
4. `ga4_sessions` — website sessions observed only where GA4 data is available.
5. `ga4_engaged_sessions` — engaged sessions observed only where GA4 data is available.

`report_date`, `client_hash_id`, and `content_hash_id` are retained only as context fields for grouping, joining, and future time-based label construction. They are not model features.

In [10]:
feature_frame_check = feature_frame[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
    ]
].agg(["count", "min", "max", "mean"])

feature_frame_check

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
count,364347.000000,364347.000000,364347.0000,364347.000000,364347.000000
min,1.000000,0.000000,0.0000,0.000000,0.000000
max,39305.000000,274.000000,377.0000,792.000000,21.000000
mean,234.218259,1.084691,14.3209,3.399932,0.076776


In [11]:
position_zero_check = con.sql(f"""
    SELECT
        COUNT(*) AS usable_rows,
        COUNT(*) FILTER (
            WHERE gsc_avg_position = 0
        ) AS zero_position_rows,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE gsc_avg_position = 0
            ) / COUNT(*),
            2
        ) AS zero_position_percentage
    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""").df()

position_zero_check

,usable_rows,zero_position_rows,zero_position_percentage
0,364347,3306,0.91


### Position-quality check

Within the usable March 2026 slice, 3,306 rows, or 0.91%, have
`gsc_avg_position = 0`.

A zero position does not represent a real first-place ranking, so these rows
must not be interpreted as normal ranking observations. They should either be
excluded from position-based analysis or handled with a separate validity flag.

In [12]:
clean_feature_frame = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_engaged_sessions
    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
      AND gsc_avg_position > 0
""").df()

clean_feature_frame.shape

(361041, 8)

### Cleaned feature frame

After removing rows where `gsc_avg_position = 0`, the cleaned feature frame contains 361,041 rows.

The frame keeps exactly five model features:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `ga4_engaged_sessions`

The remaining three columns — `report_date`, `client_hash_id`, and `content_hash_id` — are context fields only.

In [7]:
APRIL_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/'
    'fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

april_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM {APRIL_REL}
""").df()

april_check

,row_count,earliest_date,latest_date
0,10424730,2026-04-01,2026-04-30


In [8]:
april_outcomes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_gsc_clicks
    FROM {APRIL_REL}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

april_outcomes.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,april_gsc_clicks
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,0.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,0.0
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,0.0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,1.0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,0.0


In [9]:
march_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS march_gsc_impressions,
        SUM(gsc_clicks) AS march_gsc_clicks,

        SUM(gsc_sum_position)
            / NULLIF(SUM(gsc_impressions), 0)
            AS march_gsc_avg_position,

        SUM(ga4_sessions) AS march_ga4_sessions,
        SUM(ga4_engaged_sessions) AS march_ga4_engaged_sessions

    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
      AND gsc_avg_position > 0

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

march_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_gsc_impressions,march_gsc_clicks,march_gsc_avg_position,march_ga4_sessions,march_ga4_engaged_sessions
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,458.0,2.0,4.338428,14.0,0.0
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3943.0,23.0,4.291149,54.0,1.0
2,client_65de48885f4ef01b,content_3c286ded8bd68120,2180.0,15.0,8.418807,30.0,2.0
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,503.0,8.0,5.727634,23.0,1.0
4,client_65de48885f4ef01b,content_ff867882e604fa96,24.0,0.0,3.583333,2.0,0.0


In [10]:
model_df = march_features.merge(
    april_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["april_click_label"] = (
    model_df["april_gsc_clicks"] > 0
).astype(int)

model_df[
    [
        "client_hash_id",
        "content_hash_id",
        "march_gsc_impressions",
        "march_gsc_clicks",
        "march_gsc_avg_position",
        "march_ga4_sessions",
        "march_ga4_engaged_sessions",
        "april_gsc_clicks",
        "april_click_label",
    ]
].head()

,client_hash_id,content_hash_id,march_gsc_impressions,march_gsc_clicks,march_gsc_avg_position,march_ga4_sessions,march_ga4_engaged_sessions,april_gsc_clicks,april_click_label
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,458.0,2.0,4.338428,14.0,0.0,8.0,1
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3943.0,23.0,4.291149,54.0,1.0,16.0,1
2,client_65de48885f4ef01b,content_3c286ded8bd68120,2180.0,15.0,8.418807,30.0,2.0,7.0,1
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,503.0,8.0,5.727634,23.0,1.0,1.0,1
4,client_65de48885f4ef01b,content_ff867882e604fa96,24.0,0.0,3.583333,2.0,0.0,4.0,1


In [11]:
label_check = (
    model_df["april_click_label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="row_count")
)

label_check["percentage"] = (
    100 * label_check["row_count"] / len(model_df)
).round(2)

print("Total joined rows:", len(model_df))
label_check

Total joined rows: 61628


,label,row_count,percentage
0,1,37236,60.42
1,0,24392,39.58


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

honest_features = [
    "march_gsc_impressions",
    "march_gsc_clicks",
    "march_gsc_avg_position",
    "march_ga4_sessions",
    "march_ga4_engaged_sessions",
]

X = model_df[honest_features]
y = model_df["april_click_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)
honest_probabilities = honest_model.predict_proba(X_test)[:, 1]

honest_accuracy = accuracy_score(y_test, honest_predictions)
honest_auc = roc_auc_score(y_test, honest_probabilities)

print(f"Honest accuracy: {honest_accuracy:.4f}")
print(f"Honest ROC-AUC: {honest_auc:.4f}")

Honest accuracy: 0.7697
Honest ROC-AUC: 0.8464


In [13]:
leakage_df = model_df.copy()

# Deliberately invalid feature:
# it directly copies the future April label.
leakage_df["leaked_april_label"] = leakage_df["april_click_label"]

leaked_features = honest_features + ["leaked_april_label"]

X_leaked = leakage_df[leaked_features]
y_leaked = leakage_df["april_click_label"]

X_train_leaked, X_test_leaked, y_train_leaked, y_test_leaked = train_test_split(
    X_leaked,
    y_leaked,
    test_size=0.20,
    random_state=42,
    stratify=y_leaked,
)

leaked_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

leaked_model.fit(X_train_leaked, y_train_leaked)

leaked_predictions = leaked_model.predict(X_test_leaked)
leaked_probabilities = leaked_model.predict_proba(X_test_leaked)[:, 1]

leaked_accuracy = accuracy_score(y_test_leaked, leaked_predictions)
leaked_auc = roc_auc_score(y_test_leaked, leaked_probabilities)

print(f"Honest accuracy: {honest_accuracy:.4f}")
print(f"Honest ROC-AUC: {honest_auc:.4f}")
print(f"Leaked accuracy: {leaked_accuracy:.4f}")
print(f"Leaked ROC-AUC: {leaked_auc:.4f}")

Honest accuracy: 0.7697
Honest ROC-AUC: 0.8464
Leaked accuracy: 1.0000
Leaked ROC-AUC: 1.0000


### Leakage experiment

The honest model used only five March 2026 features that would be available before the April outcome window.

- Honest accuracy: **0.7697**
- Honest ROC-AUC: **0.8464**

I then deliberately added `leaked_april_label`, which directly copies the future April label into the feature set.

- Leaked accuracy: **1.0000**
- Leaked ROC-AUC: **1.0000**

The perfect leaked score is not real model quality. It happens because the model is given the answer it is supposed to predict.

Therefore, `leaked_april_label`, `april_gsc_clicks`, and all other April outcome information are excluded from the final feature set. 
The valid retained result is the honest ROC-AUC of **0.8464**.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This data contract supports directional decision-making, but it cannot establish causation.

The main limitations are:

- Client history is unbalanced. Different clients begin providing GSC and GA4 data on different dates, so not every client has the same amount of usable history.
- The analysis includes only rows where both `gsc_data_available IS TRUE` and `ga4_data_available IS TRUE`. This improves measurement quality but may introduce selection bias because clients without both sources are excluded.
- The March 2026 feature window is only one month. Performance patterns observed in March may not generalize to other months or seasonal conditions.
- `gsc_avg_position = 0` is treated as invalid and removed. This may exclude content with incomplete search measurements.
- GSC and GA4 metrics describe observed traffic and engagement, but they cannot explain why performance changed.
- The hashed client and content identifiers provide no semantic information and must not be used as model features.
- June 2026 remains a sealed future evaluation month and is not used for feature development or label experimentation.

Therefore, the output should be interpreted as decision support for estimating whether content will receive at least one search click in April, not as proof that a particular action will cause performance to improve.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x]  All required cells run successfully with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.